<a href="https://colab.research.google.com/github/FinalInvictus/Deep-Mutual-Learning-with-Semi-Aves-Dataset/blob/Sridhar/DeepMutualLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Enter your Kaggle API token: ")

Enter your Kaggle API token: ··········


In [3]:
import kagglehub

path = kagglehub.competition_download(
    "semi-inat-2020",
    output_dir="/content/semi_inat_2020",
    force_download=True
)

100%|██████████| 14.3G/14.3G [02:36<00:00, 97.7MB/s]

Extracting files...


In [ ]:
# First peer arch --> Resnet 18
# 2nd peer arch --> Resnet 18
TOTAL_CLASSES = 200
IMG_SIZE = 224
BATCH_SIZE = 32
TEMPERATURE = 1 # VALUE RANGE 1,2,4
KL_WEIGHT = 1 # VALUE RANGE 0,0.5,1,2
EPOCH = 2 # 200 WERE USED IN ORIGINAL PAPER

In [ ]:
import torch
from torch import nn

class ConvBlock(nn.Module):
  def __init__(self,in_ch,out_ch,stride=1):
    super().__init__()
    self.conv=nn.Conv2d(in_ch,out_ch,3,1,1)
    self.bnorm=nn.BatchNorm2d(out_ch)
    self.conv2=nn.Conv2d(in_ch,out_ch,3,1,1)
    self.bnorm2=nn.BatchNorm2d(out_ch)
    self.relu=nn.ReLU(inplace=True)

    self.shortcut=nn.Identity()
    if stride!=1 or in_ch!=out_ch:
      self.shortcut=nn.Sequential(
          nn.Conv2d(in_ch,out_ch,1,stride,bias=False),
          nn.BatchNorm2d(out_ch),
      )

    def forward(self,x):
      out=self.relu(self.bnorm(self.conv(x)))
      out=self.bnorm2(self.conv2(out))
      out=out+self.shortcut(x)
      return self.relu(out)


In [6]:
from google.colab import drive
drive.mount('/content/SharedData')

Mounted at /content/SharedData
